# Enterprise Sales Forecasting & Anomaly Detection EDA

This notebook is a lightweight exploratory analysis companion. The production pipeline lives in `src/`; this notebook helps explain the data patterns for interviews.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = Path('..') / 'data' / 'raw' / 'sales_data.csv'
df = pd.read_csv(DATA_PATH, parse_dates=['date'])
df.head()

## Dataset Overview

Check row count, date coverage, product/region/channel cardinality, and data types.

In [ ]:
print(df.shape)
print(df['date'].min(), df['date'].max())
print(df[['product_id', 'region', 'channel', 'product_category']].nunique())
df.describe().T

## Target Distribution

The target variable is `units_sold`. A realistic demand distribution should show spread across products and regions.

In [ ]:
df['units_sold'].hist(bins=60, figsize=(10, 4))
plt.title('Units Sold Distribution')
plt.xlabel('Units Sold')
plt.ylabel('Frequency')
plt.show()

## Anomaly Distribution

The synthetic generator injects a small percentage of business anomalies such as demand spikes, drops, abnormal revenue, and inventory issues.

In [ ]:
df['anomaly_label'].value_counts(normalize=True).rename('share')

## Sales by Region and Product Category

In [ ]:
region_sales = df.groupby('region')['units_sold'].sum().sort_values()
region_sales.plot(kind='barh', figsize=(8, 4), title='Total Units Sold by Region')
plt.show()

category_sales = df.groupby('product_category')['revenue'].sum().sort_values()
category_sales.plot(kind='barh', figsize=(8, 4), title='Total Revenue by Product Category')
plt.show()

## Seasonality

Daily demand is aggregated to reveal weekly/monthly patterns.

In [ ]:
daily = df.groupby('date')['units_sold'].sum()
daily.plot(figsize=(12, 4), title='Daily Enterprise Demand')
plt.ylabel('Units Sold')
plt.show()

df.groupby('day_of_week')['units_sold'].mean().plot(kind='bar', title='Average Demand by Day of Week')
plt.show()

df.groupby('month')['units_sold'].mean().plot(kind='bar', title='Average Demand by Month')
plt.show()

## Correlation Overview

Correlations help explain which numeric drivers are associated with demand and revenue.

In [ ]:
numeric_cols = df.select_dtypes('number').columns
corr = df[numeric_cols].corr()
corr[['units_sold', 'revenue']].sort_values('units_sold', ascending=False)